## Máster en Big Data y Data Science

### Metodologías de gestión y diseño de proyectos de big data

#### AP2 - Modelado y evaluación

---

En esta libreta se realiza la experimentación para generación del modelo de predicción objetivo del proyecto y la evaluación del mismo.
La versión del dataset a utilizar es la obtenida a partir de las operaciones de transformación.

---

En esta versión de la libreta se incorpora el registro de los detalles de la experimentación con la libreria MlFlow

In [ ]:
# Se importan las librerías necesarias y se suprimen las advertencias
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore',category=FutureWarning)
warnings.filterwarnings('ignore',category=UserWarning)

Se agrega la libreris mlflow y se configura inicialmente

In [ ]:
import mlflow
import mlflow.sklearn
from datetime import datetime

# Configuración de MLflow
mlflow.set_tracking_uri("file:.mlruns")
mlflow.set_experiment("Proyecto 13MBID-ABR2526 - Experimentación Original")

In [ ]:
# Lectura de los datos
df = pd.read_csv('../data/processed/bank-processed.csv')
df.head(5)

In [ ]:
# Se divide el dataset en variables predictoras y variable objetivo
X = df.drop('y', axis=1)
y = df['y']

In [ ]:
# Se genera el conjunto de entrenamiento y prueba con estratificación
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [ ]:
# Se separan las columnas numéricas
numerical_columns=X_train.select_dtypes(exclude='object').columns
display(numerical_columns)

categorical_columns=X_train.select_dtypes(include='object').columns
display(categorical_columns)

In [ ]:
# Se verifica la distribución de la variable objetivo en el conjunto de entrenamiento
y_train.value_counts()

In [ ]:
# Se crea un pipeline para preprocesamiento de datos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler  

# Pipeline para valores numéricos
num_pipeline = Pipeline(steps=[
    ('RobustScaler', RobustScaler())
])

# Pipeline para valores categóricos
cat_pipeline = Pipeline(steps=[
    ('OneHotEncoder', OneHotEncoder(drop='first',sparse_output=False))
])

# Se configuran los preprocesadores
preprocessor_full = ColumnTransformer([
    ('num_pipeline', num_pipeline, numerical_columns),
    ('cat_pipeline', cat_pipeline, categorical_columns)
]).set_output(transform='pandas')

In [ ]:
preprocessor_train_valid = ColumnTransformer([
    ('num_pipeline', num_pipeline, numerical_columns),
    ('cat_pipeline', cat_pipeline, categorical_columns)
]).set_output(transform='pandas')

In [ ]:
# Se ajusta y transforma el conjunto de entrenamiento y prueba
x_train_prep = preprocessor_full.fit_transform(X_train)
x_test_prep = preprocessor_full.transform(X_test)

In [ ]:
# Se aplica submuestreo a los datos preprocesados
from sklearn.utils import resample

# Combinar los datos preprocesados con las etiquetas
train_data = x_train_prep.copy()
train_data['target'] = y_train.reset_index(drop=True)

# Separar por clase
class_0 = train_data[train_data['target'] == 0]
class_1 = train_data[train_data['target'] == 1]

# Encontrar la clase minoritaria
min_count = min(len(class_0), len(class_1))

# Submuestreo balanceado - tomar una muestra igual al tamaño de la clase minoritaria
class_0_balanced = resample(class_0, n_samples=min_count, random_state=42)
class_1_balanced = resample(class_1, n_samples=min_count, random_state=42)

# Combinar las clases balanceadas
balanced_data = pd.concat([class_0_balanced, class_1_balanced])

# Separar características y objetivo
x_train_resampled = balanced_data.drop('target', axis=1)
y_train_resampled = balanced_data['target']

print(f"Tamaño original: {len(x_train_prep)}")
print(f"Tamaño balanceado: {len(x_train_resampled)}")
print(f"Distribución balanceada: {y_train_resampled.value_counts()}")

In [ ]:
from sklearn.model_selection import cross_val_score


# Se genera una función para realizar validación cruzada
def cross_val(model):
    scores = cross_val_score(model,x_train_resampled , y_train_resampled, cv=5, scoring='f1')
    print('cross validation f1 scores',scores*100)
    print('cross validation f1 mean',scores.mean()*100)
    print('cross validation f1 std',scores.std())
    print('-'*50)
    scores = cross_val_score(model,x_train_resampled , y_train_resampled, cv=5, scoring='recall')
    print('cross validation recall scores',scores*100)
    print('cross validation recall mean',scores.mean()*100)
    print('cross validation recall std',scores.std())

---
Este apartado se va a cambiar

In [ ]:
# Se aplica un modelo de regresión logística
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C=1,penalty='l2',solver='liblinear',random_state=1,max_iter=100,tol=0.000000001)

# cross validation scores
cross_val(lr)

---
El cambio consiste en que la función directamente registre el log de la experimentación  en mlflow

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from mlflow.models import infer_signature

def cross_val_mlflow(model, model_name, params=None):
    """
    Realiza validación cruzada  registra los resultados en MLFlow
    """
    with mlflow.start_run(run_name=model_name):
        f1_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='f1')
        f1_mean = f1_scores.mean()
        f1_std = f1_scores.std()

        recall_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='recall')
        recall_mean = recall_scores.mean()
        recall_std = recall_scores.std()    

        precision_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='precision')
        precision_mean = precision_scores.mean()
        precision_std = precision_scores.std()    

        accuracy_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
        accuracy_mean = accuracy_scores.mean()
        accuracy_std = accuracy_scores.std()    

        # Entrenar al modelo
        model.fit(x_train_resampled, y_train_resampled)

        # Realizar predicciones
        y_pred = model.predict(x_test_prep)

        # Obtener model signature
        signature = infer_signature(y_test, y_pred)
        test_f1 = f1_score(y_test, y_pred)
        test_recall = recall_score(y_test, y_pred)
        test_precision = precision_score(y_test, y_pred)
        test_accuracy = accuracy_score(y_test, y_pred)

        # Registramos los parámetros y métricas e MLflow
        if params:
            mlflow.log_params(params)
        else:
            mlflow.log_params(model.get_params())
        
        mlflow.log_params({
            "train_samples": len(x_train_resampled), 
            "test_samples": len(x_test_prep), 
            "balancing_method": "undersampling",
            "cvfolds": 5
        })

        # Registramos las métricas de validación cruzada
        mlflow.log_metrics({
            "CV_f1_mean" : f1_mean, 
            "CV_f1_std" : f1_std, 
            "CV_recall_mean" : recall_mean, 
            "CV_recall_std" : recall_std,             
            "CV_precision_mean" : precision_mean, 
            "CV_precision_std" : precision_std, 
            "CV_accuracy_mean" : accuracy_mean, 
            "CV_accuracy_std" : accuracy_std, 
        })

        # Registramos las métricas de test
        mlflow.log_metrics({
            "test_f1" : test_f1, 
            "test_recall" : test_recall, 
            "test_precision" : test_precision, 
            "test_accuracy" : test_accuracy, 
        })


        # Registramos el modelo
        mlflow.sklearn.log_model(
            model, 
            artifact_path= "model", 
            signature = signature

        )

        print(f"Modelo {model_name} registrado en MLFlow con ID de ejecución: {mlflow.active_run().info.run_id}")

        return model, {
            "cv_f1_mean": f1_mean,
            "cv_recall_mean": recall_mean,
            "cv_precision_mean": precision_mean,
            "cv_accuracy_mean": accuracy_mean,
            "test_f1": test_f1,
            "test_recall": test_recall,
            "test_precision": test_precision,   
            "test_accuracy": test_accuracy
        }

-----

Se cambian estas celdas por una sola que hace todas las llamadas

In [ ]:
# LinearSVC
from sklearn.svm import LinearSVC
svc = LinearSVC(max_iter=10000,tol=0.001)

# cross validation scores
cross_val(svc)

In [ ]:
# knclassifier
from sklearn.neighbors import KNeighborsClassifier
knc = KNeighborsClassifier(n_neighbors=7)

# cross validation scores
cross_val(knc)

In [ ]:
# decision tree classifier
from sklearn.tree import DecisionTreeClassifier
tree=DecisionTreeClassifier()

# cross validation scores
cross_val(tree)

In [ ]:
# decision tree plot
from sklearn.tree import plot_tree
tree.fit(x_train_prep, y_train)
plot_tree(tree, filled=True, rounded=True,max_depth=2,fontsize=10)
plt.show()

In [ ]:
# Se obtiene la matriz de confusión para el modelo
from sklearn.metrics import confusion_matrix

y_pred = tree.predict(x_test_prep)
cm = confusion_matrix(y_test, y_pred)
print(cm)


In [ ]:
# Se visualiza la matriz de confusión
from sklearn.metrics import ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['no', 'yes'])
disp.plot()

----

Esta es la nueva celda que hace la invocación al proceso con mlflow

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
import xgboost as xgb
from xgboost import XGBClassifier

# Resultados
resultados = {}

# Método 1 - Regresión
lr = LogisticRegression(C=1,penalty='l2',solver='liblinear',random_state=1,max_iter=100,tol=0.000000001)
model_lr, metrics_lr = cross_val_mlflow(lr, "Logistic Regression")
resultados['Logistic Regression'] = metrics_lr

# Método 2 - SVC
svc = LinearSVC(max_iter=10000,tol=0.001)
model_svc, metrics_svc = cross_val_mlflow(svc, "Linear SVC")
resultados['Linear SVC'] = metrics_svc

# Método 3 - KNN
knc = KNeighborsClassifier(n_neighbors=7)
model_knc, metrics_knc = cross_val_mlflow(knc, "K-Nearest Neighbors")
resultados['K-Nearest Neighbors'] = metrics_knc

# Método 4 - Decision Tree
tree = DecisionTreeClassifier()
model_tree, metrics_tree = cross_val_mlflow(tree, "Decision Tree Classifier")
resultados['Decision Tree Classifier'] = metrics_tree


# Método 5 - Gradient Boosting
gBost = XGBClassifier()
model_gbost, metrics_gBost = cross_val_mlflow(gBost, "Gradient Boosting")
resultados['Gradient Boosting'] = metrics_gBost




A continuación, se realiza un comparativa de los los modelos con la finalidad de seleccionar el que será utilizado posteriormente.

In [ ]:
df_comparacion = pd.DataFrame(resultados).T
df_comparacion = df_comparacion.round(4)
df_comparacion = df_comparacion.sort_values(by='test_f1', ascending=False)

display(df_comparacion)

print("\nEl mejor modelo basado en F1 en el conjunto de prueba es:", df_comparacion.index[0])
print(f"Valor de F1 en test: {df_comparacion.iloc[0]['test_f1']}")
print(f"Valor de recall en test: {df_comparacion.iloc[0]['test_recall']}")

#### Predicción con datos nuevos (sin clasificar)

In [ ]:
df_nuevos = pd.read_csv('../data/raw/bank-additional-new.csv')
df_nuevos.head(5)

In [ ]:
# Diagnosticar el problema con los nuevos datos
print("Información del conjunto de datos nuevos:")
print(f"Forma: {df_nuevos.shape}")
print("\nTipos de datos:")
print(df_nuevos.dtypes)
print("\nValores nulos:")
print(df_nuevos.isnull().sum())
print("\nColumnas categóricas en nuevos datos:")
print(df_nuevos.select_dtypes(include='object').columns.tolist())
print("\nColumnas numéricas en nuevos datos:")
print(df_nuevos.select_dtypes(exclude='object').columns.tolist())

In [ ]:
# Comparar con los datos de entrenamiento originales
print("Comparación de columnas:")
print(f"Columnas en datos originales: {list(X.columns)}")
print(f"Columnas en datos nuevos: {list(df_nuevos.columns)}")

print("\nColumnas que están en nuevos pero no en originales:")
new_cols = set(df_nuevos.columns) - set(X.columns)
print(new_cols)

print("\nColumnas que están en originales pero no en nuevos:")
missing_cols = set(X.columns) - set(df_nuevos.columns)
print(missing_cols)

In [ ]:
# Se hace la predicción con los nuevos datos
# Primero, eliminar la columna objetivo si existe y preparar las características
X_new = df_nuevos.drop('y', axis=1) if 'y' in df_nuevos.columns else df_nuevos.copy()

# Asegurar que las columnas estén en el mismo orden que en el entrenamiento
X_new = X_new[X.columns]

# Manejar la columna contacted_before para que coincida con el formato de entrenamiento
# En entrenamiento: 'no', 'yes' (string)
# En nuevos datos: NaN -> necesita convertirse a 'no' (asumiendo que NaN significa no contactado)
X_new['contacted_before'] = X_new['contacted_before'].fillna('no')

# Convertir cualquier valor numérico a string si es necesario
if X_new['contacted_before'].dtype in ['float64', 'int64']:
    X_new['contacted_before'] = X_new['contacted_before'].map({0.0: 'no', 1.0: 'yes'}).fillna('no')

# Asegurar que contacted_before sea de tipo object como en entrenamiento
X_new['contacted_before'] = X_new['contacted_before'].astype('object')

# Transformar los nuevos datos usando el mismo preprocesador y predecir
try:
    x_new_prep = preprocessor_full.transform(X_new)
    
    y_new_pred = tree.predict(x_new_prep)
    print(f"\nPredicciones: {y_new_pred}")
    
    predictions_df = pd.DataFrame({
        'Cliente': range(1, len(y_new_pred) + 1),
        'Predicción_Numérica': y_new_pred,
        'Suscribirá': ['No' if pred == 0 else 'Sí' for pred in y_new_pred]
    })
    print("\nResultados detallados:")
    print(predictions_df.to_string(index=False))
    
    # Resumen de predicciones
    pred_counts = pd.Series(y_new_pred).value_counts()
    print("\nResumen de predicciones:")
    for pred_val, count in pred_counts.items():
        label = 'No realizará un depósito' if pred_val == 0 else 'Sí realizará un depósito'
        print(f"  {label}: {count} clientes ({count/len(y_new_pred)*100:.1f}%)")
    
except Exception as e:
    print(f"Error durante el preprocesamiento o predicción: {e}")
    print("Información adicional para depuración:")
    print(f"Tipos de datos en X_new:\n{X_new.dtypes}")